# Homework 3: Coding

In the code spaces below, write code that answers each question. You can assume that the code spaces will all be executed in order, so any necessary libraries only need to be imported once. 

In [18]:
import math # You'll need this for pi


### Phase gates and rotation around the Z axis

In our lecture on single-qubit gates we talked about the Pauli gates which represent 180° rotations around the X, Y, and Z axes of the Bloch sphere. Rotation around the Z axis does not directly affect the probability of the 0 and 1 measurement outcomes in the computational basis, but relates to a position around the complex plane (the *phase* of the state). 

Several related gates are often defined. The S gate and T gates are defined as partial rotations around the Z axis. The S gate behaves like half of a Z gate, rotating 90° around the Z axis and the T gate behaves like a quarter of a Z gate, rotating 45° around the Z axis. 

For the exercises below, you'll use a phase gate from Qiskit to mimic these gates. Remember that rotations passed to the `p` method are in radians, not degrees. 

#### Exercise 1 
   Use the gates described in the `demo_single_qubit_circuits.ipynb` to implement a circuit equivalent to the following circuit: 
   
![quirk1](./img/img1.png)

Specifically, use the phase gate (`p`) to represent the S gate shown. Print out the state coefficients for each outcome, then print out the absolute values of the state coefficients, then print out the squares of the absolute values of the state coefficients, and verify that the probabilities of the outcomes are as they appear to be in the illustration. 

In [7]:

import math
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

circuit = QuantumCircuit(1)

circuit.h(0)
circuit.p(math.pi/2, 0)
circuit.h(0)

state = Statevector(circuit)

print(state.data)
print(abs(state.data))
print(abs(state.data)**2)

[0.5+0.5j 0.5-0.5j]
[0.70710678 0.70710678]
[0.5 0.5]


#### Exercise 2 
Use the gates described in the `demo_single_qubit_circuits.ipynb` to implement a circuit equivalent to the following circuit: 

![quirk2](./img/img2.png)

Similarly to what you did above, use the `p` method to represent the behavior of the T gate. As you did previously, print out the state coefficients for each outcome, the absolute values of the state coefficients, and the squares of the absolute values of the state coefficients, and verify that the probabilities of the outcomes are as they appear to be in the illustration.

In [ ]:
circuit = QuantumCircuit(1)
circuit.h(0)
circuit.p(math.pi/4, 0)
circuit.h(0)
state=Statevector(circuit)
print(state.data)
print(abs(state.data))
print(abs(state.data)**2)

[0.85355339+0.35355339j 0.14644661-0.35355339j]
[0.92387953 0.38268343]
[0.85355339 0.14644661]


#### Exercise 3
Using any combination of **Hadamard** gates and **Phase** gates, make a circuit such that the state is flipped from $|0\rangle$ (the default start state) to $|1\rangle$. Conclude your code for this exercise with the `state.draw('latex')` command to verify that the resulting state is $|1\rangle$ 

In [9]:
circuit.h(0)
circuit.p(math.pi, 0)
circuit.h(0)
state=Statevector(circuit)
state.draw('latex')

<IPython.core.display.Latex object>

#### Exercise 4
Find a sequence of Hadamard and Phase gates (Z, S, or T gates implemented as P gates) that results in a .25 probability of measuring 1 and a .75 probability of measuring zero. Verify this by printing out both probabilities at the end of your answer. 

In [ ]:
circuit.h(0)
circuit.p(math.pi/3, 0)
circuit.h(0)
state=Statevector(circuit)
print(round(abs(state[0])**2, 2))
print(round(abs(state[1])**2, 2))

0.85
0.15


#### Exercise 5

Write a circuit in Qiskit using H and P gates that results in the state $|i\rangle$. Call `state.draw('latex')` at the end of your solution to print out the state in braket notation and verify that it is correct. 

It will help to remember that $\frac{1}{\sqrt{2}}=\frac{\sqrt{2}}{2}$. In some contexts it is customary to 'rationalize the denominator' to eliminate radicals in the dnenominator. Qiskit generally does this for us in cases like this example, but I (and a lot of quantum textbooks) often forgo this step, since it doesn't really clarify the important thing about the fraction, which is that squaring it gives us $\frac{1}{2}$. So the output you get here may not be identical to the representation of $|i\rangle$ that you have been shown in class, but it is straightforward to show that it is equivalent. 


In [14]:
circuit = QuantumCircuit(1)
circuit.h(0)
circuit.p(math.pi / 2, 0)
state = Statevector(circuit)
state.draw('latex')

<IPython.core.display.Latex object>

#### Exercise 6

Run your solution to Exercise 2 using an Aer simulator backend. Use 1000 shots. Print out the counts and confirm that you have gotten measurments of 1 approximately 15% of the time and 0 approximately 85% of the time. If you initialize the AerSimulator backend without setting a seed you will get different results each time. Do this and run the code a few times to convince yourself that the output is conforming to the expected probabilities. 

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

circuit=QuantumCircuit(1, 1)
circuit.h(0)
circuit.p(math.pi / 4, 0)
circuit.h(0)
circuit.measure(0, 0)

simulator = AerSimulator()
result = simulator.run(transpile(circuit, simulator), shots=1000).result()
print(result.get_counts())

{'0': 854, '1': 146}


In [30]:
import os
from dotenv import load_dotenv
from qiskit_ibm_runtime import QiskitRuntimeService

load_dotenv()
token = os.getenv("IBMQ_API_KEY")


#### Exercise 7

Run your solution to Exercise 2 using an IBM quantum backend. Print out the backend so you know what the name of the machine is that the workload is running on, and print out the result counts. Use 100 shots. 

In [38]:
circuit = QuantumCircuit(1,1)

circuit.h(0)
circuit.p(math.pi/4, 0)
circuit.h(0)
circuit.measure(0, 0)

In [37]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler

if not token:
    raise ValueError('Set IBMQ_API_KEY in your .env file')
service=QiskitRuntimeService(channel='ibm_quantum_platform', token=token)
backend=service.least_busy(operational=True, simulator=False)
print(backend.name)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
sampler = Sampler(mode=backend)
job =sampler.run([isa_circuit], shots=100)
result = job.result()[0]
print(result.data.c.get_counts())

qiskit_runtime_service._discover_account:WARNING:2026-11-10 09:12:27,163: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-11-10 09:12:30,250: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-11-10 09:12:30,605: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-10 09:12:32,172: Using instance: open-instance, plan: open


ibm_fez


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_15602/2322141346.py:12: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


{'0': 86, '1': 14}


#### Exercise 8

Run your solution to Exercise 5 using an Aer simulator backend. Use 100 shots. Print out the counts and confirm that you have gotten measurments of 1 approximately 50% of the time and 0 approximately 50% of the time. Use 100 shots.

In [49]:
circuit = QuantumCircuit(1, 1)
circuit.h(0)
circuit.p(math.pi / 2, 0)
circuit.measure(0, 0)
result = simulator.run(transpile(circuit,simulator), shots=100).result()
print(result.get_counts())

{'0': 46, '1': 54}


#### Exercise 9

Run your solution to Exercise 5 using an IBM quantum backend. Print out the backend so you know what the name of the machine is that the workload is running on, and print out the result counts. Use 100 shots.

In [50]:
backend = service.least_busy(operational=True, simulator=False)
print(backend.name)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
sampler=Sampler(mode=backend)
job=sampler.run([isa_circuit], shots=100)
result=job.result()[0]
print(result.data.c.get_counts())

qiskit_runtime_service.backends:WARNING:2026-11-10 09:17:42,015: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-10 09:17:42,882: Using instance: open-instance, plan: open


ibm_fez


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_15602/966604770.py:5: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler=Sampler(mode=backend)


{'1': 46, '0': 54}


#### Exercise 10

Using H and P gates, create a circuit that will generate 1s with .75 chance and 0s with .25 chance. Run this on IBM quantum hardware to get a truly random sequence according to this probability distribution. Print out the counts and the bitstrings. for 100 shots 

In [51]:
circuit = QuantumCircuit(1, 1)
circuit.h(0)
circuit.p(2 * math.pi / 3, 0)
circuit.h(0)
circuit.measure(0, 0)

backend = service.least_busy(operational=True,simulator=False)
print(backend.name)
pm = generate_preset_pass_manager(backend=backend,optimization_level=1)
isa_circuit = pm.run(circuit)
sampler = Sampler(mode=backend)
job = sampler.run([isa_circuit], shots=100)
result = job.result()[0]
counts = result.data.c.get_counts()
print(counts)
print(result.data.c.get_bitstrings())

qiskit_runtime_service.backends:WARNING:2026-11-10 09:17:56,992: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-10 09:17:57,525: Using instance: open-instance, plan: open


ibm_fez


/var/folders/8y/t4fp66zs4pn5dgw11026qw580000gn/T/ipykernel_15602/39683979.py:11: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = Sampler(mode=backend)


{'1': 72, '0': 28}
['1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '0', '1', '1', '0', '0', '1', '1', '1', '1', '0', '1', '0', '0', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '1', '1', '0', '1', '1', '0', '1', '0', '0', '1', '1', '1', '1', '0', '1', '0', '0', '0', '1', '1', '0', '0', '1', '1', '1', '1', '1', '1', '0', '1', '1', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0']
